# Drone mapping simulation

1. Create drone flight-path coordinates
2. Evaluate a beam model at those (scattered, non-gridded) coordinates
3. Grid the values onto a regular array for visualization

Run this notebook from the repository root. Step 2.2 reads
`outputs/coefficients_400.csv` / `outputs/scaleparameters_400.csv`, which
`beamlab21 fit configs/config_fit.yaml` writes.

## 1. Drone path

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

from beamlab21.drone import create_drone_path, evaluate_gaussian_on_path, evaluate_zernike_on_path

# Zigzag flight path over a 150m x 150m area
coords = create_drone_path(
    width=150.0, height=150.0, dx=4, dy=1, ds=0.5, jitter=0.92, direction='NS'
)

plt.plot(coords[:, 0], coords[:, 1], marker='o', ms=0.5)
plt.gca().set_aspect('equal')
plt.show()

## 2. Evaluate beam
### 2.1 Gaussian beam

In [ ]:
xdr, ydr = coords[:, 0] - (coords[:, 0].max() / 2), coords[:, 1] - (coords[:, 1].max() / 2)

amp = 1
xo, yo = 0, 0
sigx, sigy = 4.25, 5.1  # fwhm/(2*sqrt(2*ln2)) for fwhmx=10, fwhmy=12
tilt = 0  # degrees

gaussian_beam = evaluate_gaussian_on_path(xdr, ydr, (amp, sigx, sigy, xo, yo, tilt))

In [ ]:
plt.scatter(xdr, ydr, c=gaussian_beam, s=1.5, marker=',')
plt.show()

### 2.2 Zernike beam

Needs coefficients/scale parameters from a previous `beamlab21 fit` run:
```bash
beamlab21 fit configs/config_fit.yaml
```

In [ ]:
coeffile = 'outputs/coefficients_400.csv'
spfile = 'outputs/scaleparameters_400.csv'

zernike_beam = evaluate_zernike_on_path(xdr, ydr, coeffile, spfile)

In [ ]:
plt.scatter(xdr, ydr, c=np.log10(zernike_beam), s=1.5, marker=',')
plt.show()

## 3. Gridding

In [ ]:
from scipy.interpolate import griddata

xnew, ynew = np.linspace(xdr.min(), xdr.max(), 300), np.linspace(ydr.min(), ydr.max(), 300)

gridded_drone_zernike_beam = griddata(
    (xdr, ydr), zernike_beam, (xnew[None, :], ynew[:, None]), method='cubic'
)
gridded_drone_zernike_beam.shape

In [ ]:
plt.pcolormesh(xnew, ynew, np.log10(gridded_drone_zernike_beam), cmap='viridis')
plt.show()